In [1]:
# 1. setup
from leelax.net.model import LeelaXNet
from leelax.selfplay.worker import SelfPlayWorker
from leelax.selfplay.replay_buffer import ReplayBuffer
from leelax.train.loop import train_for_n_steps
from leelax.train.logger import create_tb_writer
import torch

model = LeelaXNet()
device = "cpu"

def net_fn(x):
    with torch.no_grad():
        return model(x.to(device))

worker = SelfPlayWorker(net_fn, n_simulations=32, temperature=1.0, device=device)
buffer = ReplayBuffer(capacity=50_000)

In [2]:
# 2. generate games
for _ in range(5):
    samples, moves = worker.play_game(verbose=False)
    buffer.add_many(samples)
len(buffer)

AttributeError: 'GameRecorder' object has no attribute 'finalize'

In [ ]:
# 3. train
writer = create_tb_writer("runs/notebook")
train_for_n_steps(model, buffer, n_steps=100, device=device, tb_writer=writer)